# Stage B v7 — Graph-Augmented Swiss Citation Expert

**Paradigm**: v6 continuous scoring + Pass-1 issue map, **plus** bibliographic-coupling graph expansion from the existing 5.3M-edge citation graph in `unified_retrieval.sqlite`.

**Proven locally on val_009**: bibliographic coupling from a 4-article Pass-1 landscape recovers 8 of 10 gold (Art. 277, 286, 291 ZGB caught via co-citation that v5.x missed). Architecture mirrors COLIEE 2025 winners (CaseLink GNN, NOWJ multi-stage).

**Inputs**: needs 3 graph parquets pre-computed locally and uploaded to Drive:
- `graph_chapter_neighbors.parquet`  (71k citations, 1.0 MB)
- `graph_statute_to_precedents.parquet` (183k statutes, 21.5 MB)
- `graph_paragraph_cites.parquet` (1.55M paragraphs, 82.9 MB)

**v7 target**: macro F1 0.55 - 0.65 (v6 projected ~0.45, v5.3 measured 0.107).

## Phase 0 — Setup

In [ ]:
import os, sys, subprocess, json, time, gc, io, re, math
from pathlib import Path
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")

IS_COLAB = "google.colab" in sys.modules
print(f"Colab: {IS_COLAB}")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run(["pip","install","-q","-U",
        "vllm>=0.9.1","transformers>=4.51.0","pandas==2.2.3",
        "pyarrow==16.1.0","numpy==1.26.4","tqdm"], check=True)


## Phase 1 — Paths and config

In [ ]:
DRIVE_ROOT  = Path("/content/drive/MyDrive/swiss_law")
STAGE_B_IN  = DRIVE_ROOT / "research" / "stage_b_input" / "stage_b_input.parquet"
VAL_CSV     = DRIVE_ROOT / "data"     / "val.csv"
GOLD_SETS   = DRIVE_ROOT / "research" / "anchor_funnel_val001_v7" / "snapshot" / "gold_doc_sets.json"
VAL_ASPECTS = DRIVE_ROOT / "research" / "concept_embedding_path" / "multi_aspect" / "val_aspects.parquet"

# Graph indices (pre-computed locally; upload to Drive once)
GRAPH_DIR             = DRIVE_ROOT / "research" / "stage_b_input"
GRAPH_CHAPTER         = GRAPH_DIR / "graph_chapter_neighbors.parquet"
GRAPH_STATUTE_PREC    = GRAPH_DIR / "graph_statute_to_precedents.parquet"
GRAPH_PARA_CITES      = GRAPH_DIR / "graph_paragraph_cites.parquet"

OUT_DIR = DRIVE_ROOT / "research" / "stage_b_grounded_llm" / "v7_all_queries"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K              = 2000
LLM_MODEL          = "Qwen/Qwen3-32B-AWQ"
PASS1_SEEDS        = [42, 43, 44]
PASS1_MAX_TOKENS   = 4096
PASS2_MAX_TOKENS   = 1024
PASS3_MAX_TOKENS   = 2048
MAX_MODEL_LEN      = 8192

# Graph expansion knobs
TOP_PRECEDENTS_PER_STATUTE = 30   # how many precedents per landscape statute to walk
BIB_NEIGHBORS_PER_STATUTE  = 50   # how many bibliographic neighbors per starting statute
EXPANDED_LANDSCAPE_CAP     = 250  # max statutes in expanded landscape per query

# Composite-score knobs (graph features set floors; LLM can raise)
SCORE_FLOOR        = 0.40

print(f"Verifying inputs:")
for p in [STAGE_B_IN, VAL_CSV, GOLD_SETS, VAL_ASPECTS,
          GRAPH_CHAPTER, GRAPH_STATUTE_PREC, GRAPH_PARA_CITES]:
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")


## Phase 2 — Load val data + graph indices

In [ ]:
import pandas as pd
from collections import Counter

val_df = pd.read_csv(VAL_CSV)
QIDS = sorted(val_df.query_id.unique().tolist())
qid_to_query = {r.query_id: str(r.query) for r in val_df.itertuples()}

asp_df = pd.read_parquet(VAL_ASPECTS)
qid_to_aspects = {r.query_id: list(r.aspects) for r in asp_df.itertuples()}

gold_sets = json.load(open(GOLD_SETS, encoding="utf-8"))
qid_to_gold = {q: set(gold_sets.get(q, [])) for q in QIDS}
gold_totals = {q: len(qid_to_gold[q]) for q in QIDS}

# Stage A candidates
sb_all = pd.read_parquet(STAGE_B_IN)
sb_per_query = {}
for q in QIDS:
    sub = (sb_all[sb_all.qid == q].sort_values("stage_a_rank").head(TOP_K).reset_index(drop=True))
    sub["tier"] = "drop"
    _auto = sub.article_match & ((sub.co_citation_count >= 30) | sub.code_in_target)
    sub.loc[_auto, "tier"] = "auto"
    _mid = (~_auto) & (sub.article_match | (sub.co_citation_count >= 5) | (sub.concept_cosine_score >= 0.55))
    sub.loc[_mid, "tier"] = "llm"
    sb_per_query[q] = sub

print("Loading graph indices ...")
t0 = time.time()
cn_df = pd.read_parquet(GRAPH_CHAPTER)
sp_df = pd.read_parquet(GRAPH_STATUTE_PREC)
pc_df = pd.read_parquet(GRAPH_PARA_CITES)
print(f"  chapter_neighbors:   {len(cn_df):,} rows  ({(time.time()-t0):.1f}s)")
print(f"  statute_to_precedents: {len(sp_df):,} rows")
print(f"  paragraph_cites:      {len(pc_df):,} rows")

# Convert to dict lookups (fast inner-loop access)
cn_lookup = {row.citation_norm: list(row.neighbors) for row in cn_df.itertuples()}
sp_lookup = {row.citation_norm: list(row.top_precedents)[:TOP_PRECEDENTS_PER_STATUTE]
             for row in sp_df.itertuples()}
sp_total  = {row.citation_norm: int(row.n_total_cites) for row in sp_df.itertuples()}
pc_lookup = {row.doc_id: list(row.cited_statutes_norm) for row in pc_df.itertuples()}
print(f"\nLookup dicts built in {time.time()-t0:.1f}s")

del cn_df, sp_df, pc_df
import gc; gc.collect()


## Phase 3 — Pass-1 issue map (v6 unchanged: 3 samples per query, thinking ON)

Same as v6. Multi-sample union with anti-hallucination filter.

In [ ]:
PASS1_PROMPT = """You are a senior Swiss attorney with mastery of Swiss federal law (ZGB, OR, StGB, StPO, ZPO, BGG, SchKG, BV, EMRK) and the BGE jurisprudence. Before drafting a legal opinion, you map the issues and canonical authorities.

CRITICAL CITATION RULES:
- Cite ONLY Swiss legal provisions you are confident actually exist.
- Code-size limits:  ZGB 1-977, OR 1-1186, StGB 1-393, StPO 1-457, ZPO 1-408, BGG 1-132, SchKG 1-340, BV 1-197, EMRK 1-59.
- If unsure of an article number, OMIT it. Sparse and accurate beats long and fabricated.

LEGAL QUESTION:
{question}

ANALYST'S ASPECT DECOMPOSITION (informational):
{aspects_block}

YOUR TASK:
Produce a structured issue map. For each legal ISSUE the opinion must address:
1. State the controlling rule in one sentence.
2. List MUST_CITE authorities — omission = malpractice (3-8 entries).
3. List SHOULD_CITE — thorough opinion includes (3-10 entries).
4. List PROCEDURAL_RULES — apply to any case of this procedural posture.
5. List LEADING_PRECEDENTS — specific BGE references (omit if uncertain).

OUTPUT STRICT JSON (no prose, begin with `{{`):
{{
  "issues": [{{
    "id": "i1", "label": "...", "rule_statement": "...",
    "must_cite_authorities": [{{"cit": "Art. 285 ZGB", "why": "..."}}],
    "should_cite_authorities": [{{"cit": "Art. 286 ZGB", "why": "..."}}],
    "procedural_rules": [{{"cit": "Art. 100 BGG", "why": "..."}}],
    "leading_precedents": [{{"cit": "BGE 137 III 118", "doctrine": "..."}}]
  }}]
}}
"""

def format_aspects_block(aspects):
    parts = []
    for a in aspects:
        aid, lbl = a.get("id",""), a.get("label","")
        w = float(a.get("weight",0))
        terms = list(a.get("concepts_en",[])) + list(a.get("terms_de",[]))[:4]
        parts.append(f"  {aid} (w={w:.2f}): {lbl}    [terms: {', '.join(terms)}]")
    return "\n".join(parts)

pass1_prompts, pass1_meta = [], []
for q in QIDS:
    p = PASS1_PROMPT.format(question=qid_to_query[q], aspects_block=format_aspects_block(qid_to_aspects.get(q, [])))
    for seed in PASS1_SEEDS:
        pass1_prompts.append(p); pass1_meta.append((q, seed))
print(f"Pass-1 prompts: {len(pass1_prompts)}")


## Phase 4 — Run Pass-1 (load model, batched)

In [ ]:
from vllm import LLM, SamplingParams

print(f"Loading {LLM_MODEL} ...")
llm = LLM(model=LLM_MODEL, dtype="bfloat16",
          gpu_memory_utilization=0.60, max_model_len=MAX_MODEL_LEN, enforce_eager=False)

sp_list = [SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
                          max_tokens=PASS1_MAX_TOKENS, seed=seed)
           for _, seed in pass1_meta]

print(f"Pass-1: {len(pass1_prompts)} prompts ...")
t0 = time.time()
outs1 = llm.chat([[{"role":"user","content":p}] for p in pass1_prompts], sampling_params=sp_list)
pass1_raws = [o.outputs[0].text for o in outs1]
print(f"Done in {(time.time()-t0)/60:.1f} min")


## Phase 5 — Parse + union issue maps per query

In [ ]:
def parse_json_block(raw):
    s = raw.strip()
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL).strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[-1]
        if s.endswith("```"):
            s = s.rsplit("```", 1)[0]
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1: return None
    for cand in [s[a:b+1], s[a:b+1].replace(",}", "}").replace(",]", "]")]:
        try: return json.loads(cand)
        except Exception: pass
    return None

def coerce_item(it):
    if isinstance(it, str): return {"cit": it, "why": ""}
    if isinstance(it, dict):
        return {"cit": str(it.get("cit") or it.get("citation") or it.get("article") or ""),
                "why": str(it.get("why") or it.get("doctrine") or it.get("role") or "")}
    return None

def norm_cit(c):
    s = re.sub(r"\bAbs\.\s*\d+\w*\b", "", c or "", flags=re.IGNORECASE)
    s = re.sub(r"\blit\.\s*\w+\b", "", s, flags=re.IGNORECASE)
    s = re.sub(r"[.,]", "", s)
    return " ".join(s.lower().split())

ART_LIMITS = {"ZGB":977,"OR":1186,"StGB":393,"StPO":457,"ZPO":408,
              "BGG":132,"SchKG":340,"BV":197,"EMRK":59}
ART_RE = re.compile(r"Art\.\s*(\d+)\s*(?:Abs\.\s*\d+\w*\s*)?(?:lit\.\s*\w+\s*)?(\w+)")
def is_valid(cit):
    m = ART_RE.search(cit or "")
    if not m: return True
    num, code = int(m.group(1)), m.group(2)
    return num <= ART_LIMITS.get(code, 99999)

CATS = ("must_cite_authorities","should_cite_authorities","procedural_rules","leading_precedents")

qid_to_issue_maps = {q: [] for q in QIDS}
for (qid, seed), raw in zip(pass1_meta, pass1_raws):
    p = parse_json_block(raw)
    if p and "issues" in p:
        qid_to_issue_maps[qid].append(p)
print(f"Parsed: {sum(len(v) for v in qid_to_issue_maps.values())} / {len(pass1_meta)}")

qid_to_issues = {}
for q in QIDS:
    samples = qid_to_issue_maps[q]
    if not samples:
        qid_to_issues[q] = []
        continue
    canonical = samples[0].get("issues", [])
    merged = []
    for ci in canonical:
        iid = ci.get("id","")
        ilabel = (ci.get("label","") or "").lower()
        pooled = {c: {} for c in CATS}
        for cat in CATS:
            for raw_it in (ci.get(cat, []) or []):
                it = coerce_item(raw_it)
                if not it or not it["cit"]: continue
                k = norm_cit(it["cit"])
                if k not in pooled[cat]:
                    pooled[cat][k] = it
        for other in samples[1:]:
            for oi in other.get("issues", []):
                olabel = (oi.get("label","") or "").lower()
                shared = {w for w in (set(ilabel.split()) & set(olabel.split())) if len(w) > 4}
                if not shared: continue
                for cat in CATS:
                    for raw_it in (oi.get(cat, []) or []):
                        it = coerce_item(raw_it)
                        if not it or not it["cit"]: continue
                        k = norm_cit(it["cit"])
                        if k not in pooled[cat]:
                            pooled[cat][k] = it
        final = {"id": iid, "label": ci.get("label",""), "rule_statement": ci.get("rule_statement","")}
        for cat in CATS:
            items = [it for it in pooled[cat].values() if is_valid(it.get("cit",""))]
            final[cat] = items
        merged.append(final)
    qid_to_issues[q] = merged

print("\nIssue map sizes per query (Pass-1 alone, before graph expansion):")
for q in QIDS:
    issues = qid_to_issues[q]
    n_must = sum(len(i.get("must_cite_authorities",[])) for i in issues)
    n_should = sum(len(i.get("should_cite_authorities",[])) for i in issues)
    n_proc = sum(len(i.get("procedural_rules",[])) for i in issues)
    n_prec = sum(len(i.get("leading_precedents",[])) for i in issues)
    print(f"  {q}: {len(issues)} issues, must={n_must}, should={n_should}, proc={n_proc}, prec={n_prec}")


## Phase 6 — Graph expansion (the new core)

For each query, take the union of must_cite + should_cite + procedural + precedents from Pass-1, normalize to citation strings, then for each:
1. Add 1-hop chapter neighbors via `chapter_neighbors`
2. Add bibliographic-coupled statutes via `statute_to_precedents` → `paragraph_cites`

This gives `expanded_landscape[qid]` — the set of normalized statute citations that the LLM landscape implies. Locally verified to recover 8/10 val_009 gold from a 4-article seed.

In [ ]:
def expand_landscape(issues):
    """Return (expanded_set, direct_set) of normalized citations.
    direct_set = anything literally in Pass-1.
    expanded_set = direct_set + chapter neighbors + bibliographic coupling.
    """
    direct = set()
    for iss in issues:
        for cat in CATS:
            for it in iss.get(cat, []) or []:
                c = norm_cit(it.get("cit",""))
                if c: direct.add(c)
    expanded = set(direct)
    for d in list(direct):
        # 1-hop chapter neighbors
        for nb in cn_lookup.get(d, []):
            expanded.add(nb)
        # 1-hop bibliographic coupling
        precedents = sp_lookup.get(d, [])
        co_cooc = Counter()
        for p in precedents:
            for c in pc_lookup.get(p, []):
                if c != d:
                    co_cooc[c] += 1
        for c, _ in co_cooc.most_common(BIB_NEIGHBORS_PER_STATUTE):
            expanded.add(c)
    # cap final size to keep prompt context manageable
    if len(expanded) > EXPANDED_LANDSCAPE_CAP:
        keep = set(direct)
        for d in list(direct):
            for nb in cn_lookup.get(d, [])[:5]:
                keep.add(nb)
        for d in list(direct):
            precedents = sp_lookup.get(d, [])
            co_cooc = Counter()
            for p in precedents:
                for c in pc_lookup.get(p, []):
                    if c != d:
                        co_cooc[c] += 1
            n_slots = max(5, (EXPANDED_LANDSCAPE_CAP - len(keep)) // max(1, len(direct)))
            for c, _ in co_cooc.most_common(n_slots):
                keep.add(c)
        expanded = keep
    return expanded, direct

qid_to_expanded = {}
qid_to_direct = {}
for q in QIDS:
    expanded, direct = expand_landscape(qid_to_issues.get(q, []))
    qid_to_expanded[q] = expanded
    qid_to_direct[q]   = direct
    in_topk_norm = {norm_cit(c) for c in sb_per_query[q]["citation"]}
    overlap = expanded & in_topk_norm
    gold_in_top = sb_per_query[q][sb_per_query[q].is_gold]
    gold_norm = {norm_cit(c) for c in gold_in_top["citation"]}
    gold_caught = gold_norm & expanded
    print(f"  {q}: direct={len(direct)}, expanded={len(expanded)},  "
          f"overlap with top-{TOP_K} pool: {len(overlap)},  "
          f"gold_in_topk={len(gold_norm)}, gold caught by landscape: {len(gold_caught)}")


## Phase 7 — Per-candidate graph features

For each candidate in the top-2000 pool, compute four graph signals:

- `graph_direct_match`   : citation_norm ∈ expanded_landscape  (the most reliable signal — high precision)
- `graph_co_cite_count`  : how many landscape statutes does this paragraph cite (via paragraph_cites)
- `graph_chapter_match`  : is the citation chapter-adjacent to anything in the landscape (via chapter_neighbors)
- `graph_landmark_score` : normalized popularity prior (`n_total_cites / 1000` capped at 1.0)

In [ ]:
def compute_graph_features(qid, candidate):
    expanded = qid_to_expanded.get(qid, set())
    direct   = qid_to_direct.get(qid, set())
    cit_norm = norm_cit(candidate.citation)

    direct_match = cit_norm in expanded
    chapter_match = any(nb in expanded for nb in cn_lookup.get(cit_norm, []))
    cited = set(pc_lookup.get(candidate.did, []))
    co_cite_count = len(cited & expanded)
    landmark = min(1.0, sp_total.get(cit_norm, 0) / 1000.0)
    return {
        "graph_direct_match": direct_match,
        "graph_chapter_match": chapter_match,
        "graph_co_cite_count": co_cite_count,
        "graph_landmark_score": landmark,
    }

# Compute features per candidate per query
graph_feat_per_qd = {}   # (qid, did) -> feat dict
for q in QIDS:
    sb = sb_per_query[q]
    for r in sb.itertuples():
        graph_feat_per_qd[(q, r.did)] = compute_graph_features(q, r)

# Diagnostic: how many gold have graph_direct_match?
print(f"{'qid':<8} {'gold_in_topk':>12} {'gold_direct':>11} {'gold_chapter':>12} {'gold_cocite>=1':>14}")
for q in QIDS:
    sb = sb_per_query[q]
    gold_in_topk = sb[sb.is_gold]
    n_direct, n_chapter, n_cocite = 0, 0, 0
    for r in gold_in_topk.itertuples():
        feat = graph_feat_per_qd[(q, r.did)]
        if feat["graph_direct_match"]: n_direct += 1
        if feat["graph_chapter_match"]: n_chapter += 1
        if feat["graph_co_cite_count"] >= 1: n_cocite += 1
    print(f"  {q:<6} {len(gold_in_topk):>12} {n_direct:>11} {n_chapter:>12} {n_cocite:>14}")


## Phase 8 — Build Pass-2 prompts with graph evidence injected

Same continuous-scoring prompt as v6, with one new section: **GRAPH EVIDENCE** showing the model what the citation graph says about this candidate.

In [ ]:
def render_issue_map(issues):
    if not issues: return "(no issue map)"
    chunks = []
    for iss in issues:
        chunks.append(f"== Issue {iss.get('id','?')}: {iss.get('label','')} ==")
        rs = iss.get("rule_statement","")
        if rs: chunks.append(f"  Rule: {rs}")
        for cat, hdr in [("must_cite_authorities","MUST-CITE (malpractice if omitted)"),
                         ("should_cite_authorities","Should-cite (thorough opinion)"),
                         ("procedural_rules","Procedural rules"),
                         ("leading_precedents","Leading BGE precedents")]:
            items = iss.get(cat, [])
            if not items: continue
            chunks.append(f"  {hdr}:")
            for it in items:
                chunks.append(f"    - {it.get('cit',''):<25}  {it.get('why','')[:100]}")
    return "\n".join(chunks)

PASS2_PROMPT = """You are a senior Swiss lawyer evaluating one candidate citation for inclusion in your legal opinion. You have already mapped the issues and canonical authorities. Now score this candidate on three continuous dimensions.

LEGAL QUESTION:
{question}

ISSUE MAP (your prior research):
{issue_map}

CANDIDATE:
- Citation: {citation}
- Type: {family_label}  ({family_extra})
- Paragraph role: {role}
- Substantive text (original language):
---BEGIN---
{text}
---END---

CITATION-GRAPH EVIDENCE (objective, computed from 5.3M citation edges in Swiss legal corpus):
- Direct match: candidate citation appears in your expanded landscape: {graph_direct}
- Chapter match: citation is chapter-adjacent to a landscape statute: {graph_chapter}
- Co-citation count: this paragraph cites {graph_cocite} statutes from your landscape
- Popularity: this article is cited {landmark_cites:,} times across all Swiss court paragraphs

RETRIEVAL EVIDENCE (advisory):
- Query names this article: {article_match}  | co-cit in pool: {co_citation_count}  | concept-cosine: {concept_cosine_score:.2f}

SCORING DIMENSIONS (each independently 0.0-1.0):

A. must_cite_score — Would OMITTING this be a serious defect?
   1.0 = foundational/controlling for an issue (citation literally in MUST-CITE, or text states controlling rule)
   0.7 = strong overlap
   0.4 = arguable
   0.0 = no

B. should_cite_score — Would a thorough opinion include this?
   1.0 = yes (listed SHOULD-CITE or text strongly supports an issue)
   0.5 = peripheral
   0.0 = no

C. supporting_relevance — Does the text directly serve a legal argument the opinion makes?
   1.0 = states a rule/holding the opinion will rely on
   0.5 = mentions topic but doesn't state rule
   0.0 = unrelated

BE CALIBRATED. Use the graph evidence as strong input. If graph_direct=true, must_cite_score should usually be >= 0.85. If graph_cocite >= 2, must or should should be >= 0.7. If text matches the rule_statement, score high regardless of graph.

OUTPUT STRICT JSON (no prose):
{{"must_cite_score": <0.0-1.0>, "should_cite_score": <0.0-1.0>, "supporting_relevance": <0.0-1.0>, "matched_issue": "<i1|i2|...|none>", "citation_role": "<foundational|procedural|supporting|tangential|off_topic>", "reasoning": "<one sentence>"}}
"""

def family_label(r): return "Swiss court precedent paragraph" if r.family=="court" else "Swiss statutory provision"
def family_extra(r):
    if r.family=="court": return f"Court base: {r.court_base!s}, Chamber: {r.chamber!s}"
    return f"Code: {r.law_code!s}, Law: {r.law_title!s}"

def build_pass2(r, q, issue_map_text, feat):
    return PASS2_PROMPT.format(
        question=qid_to_query[q][:1500],
        issue_map=issue_map_text,
        citation=r.citation,
        family_label=family_label(r),
        family_extra=family_extra(r),
        role=r.role or "(unknown)",
        graph_direct=str(feat["graph_direct_match"]).lower(),
        graph_chapter=str(feat["graph_chapter_match"]).lower(),
        graph_cocite=int(feat["graph_co_cite_count"]),
        landmark_cites=int(sp_total.get(norm_cit(r.citation), 0)),
        article_match=str(bool(r.article_match)).lower(),
        co_citation_count=int(r.co_citation_count),
        concept_cosine_score=float(r.concept_cosine_score),
        text=(r.text or "")[:1200].replace('"', "'"),
    )

pass2_prompts, pass2_meta = [], []
for q in QIDS:
    issue_map_text = render_issue_map(qid_to_issues.get(q, []))
    sb_llm = sb_per_query[q][sb_per_query[q].tier == "llm"].reset_index(drop=True)
    for r in sb_llm.itertuples():
        feat = graph_feat_per_qd[(q, r.did)]
        pass2_prompts.append(build_pass2(r, q, issue_map_text, feat))
        pass2_meta.append((q, r.did))
print(f"Pass-2 prompts: {len(pass2_prompts)}  mean_len={int(sum(len(p) for p in pass2_prompts)/max(1,len(pass2_prompts)))}")


## Phase 9 — Run Pass-2

In [ ]:
sp_pass2 = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
                          max_tokens=PASS2_MAX_TOKENS, seed=42)

print(f"Pass-2: {len(pass2_prompts):,} prompts ...")
t0 = time.time()
outs2 = llm.chat([[{"role":"user","content":p}] for p in pass2_prompts], sampling_params=sp_pass2)
pass2_raws = [o.outputs[0].text for o in outs2]
print(f"Done in {(time.time()-t0)/60:.1f} min")


## Phase 10 — Parse Pass-2 + graph-aware composite

Composite formula:
```
composite = max(
    0.95 if graph_direct_match,
    0.90 if graph_co_cite_count >= 2,
    0.85 if (graph_chapter_match AND llm_must_cite >= 0.4),
    llm_must_cite_score,
    0.7 * llm_should_cite_score,
    0.4 * llm_supporting_relevance,
)
```
Graph signals set floors; LLM can RAISE the score with strong text evidence but cannot reject a foundational graph match.

In [ ]:
def parse_scores(raw):
    parsed = parse_json_block(raw) or {}
    def f(k, d=0.0):
        try: return float(parsed.get(k, d) or d)
        except Exception: return d
    must = max(0.0, min(1.0, f("must_cite_score")))
    should = max(0.0, min(1.0, f("should_cite_score")))
    support = max(0.0, min(1.0, f("supporting_relevance")))
    return {
        "must_cite_score": must, "should_cite_score": should, "supporting_relevance": support,
        "matched_issue": str(parsed.get("matched_issue","") or "")[:8],
        "citation_role": str(parsed.get("citation_role","") or "")[:30],
        "reasoning": str(parsed.get("reasoning","") or "")[:300],
        "parse_ok": bool(parsed),
    }

def graph_aware_composite(scores, feat):
    candidates = [
        scores["must_cite_score"],
        0.7 * scores["should_cite_score"],
        0.4 * scores["supporting_relevance"],
    ]
    if feat["graph_direct_match"]:                                    candidates.append(0.95)
    if feat["graph_co_cite_count"] >= 2:                              candidates.append(0.90)
    if feat["graph_chapter_match"] and scores["must_cite_score"] >= 0.4:  candidates.append(0.85)
    return max(candidates)

def auto_composite(cc):
    return min(0.70, 0.50 + 0.20 * math.log(max(1, cc) + 1) / 10.0)

pass2_lookup = {(q, d): r for (q, d), r in zip(pass2_meta, pass2_raws)}

all_rows = []
for q in QIDS:
    sb = sb_per_query[q]
    for r in sb[sb.tier=="auto"].itertuples():
        feat = graph_feat_per_qd[(q, r.did)]
        # AUTO tier: composite is dossier-based plus graph boost
        base = auto_composite(int(r.co_citation_count))
        if feat["graph_direct_match"]:    base = max(base, 0.95)
        elif feat["graph_co_cite_count"] >= 2:  base = max(base, 0.90)
        elif feat["graph_chapter_match"]:       base = max(base, 0.80)
        all_rows.append({
            "qid": q, "did": r.did, "is_gold": bool(r.is_gold), "tier": "auto",
            "citation": r.citation, "family": r.family, "stage_a_rank": int(r.stage_a_rank),
            "must_cite_score": 0.0, "should_cite_score": 0.0, "supporting_relevance": 0.0,
            "composite": base, "matched_issue": "auto", "citation_role": "auto_dossier",
            "reasoning": f"dossier cc={int(r.co_citation_count)}", "parse_ok": True,
            **feat,
        })
    for r in sb[sb.tier=="llm"].itertuples():
        feat = graph_feat_per_qd[(q, r.did)]
        scores = parse_scores(pass2_lookup.get((q, r.did), ""))
        composite = graph_aware_composite(scores, feat)
        all_rows.append({
            "qid": q, "did": r.did, "is_gold": bool(r.is_gold), "tier": "llm",
            "citation": r.citation, "family": r.family, "stage_a_rank": int(r.stage_a_rank),
            **scores, "composite": composite,
            **feat,
        })

out_df = pd.DataFrame(all_rows)
print(f"Total rows: {len(out_df)}")
print(f"\nPer-tier composite distribution:")
print(out_df.groupby("tier")["composite"].describe()[["mean","50%","75%","max"]])

gold = out_df[out_df.is_gold]; non = out_df[~out_df.is_gold]
print(f"\nGold mean composite:     {gold.composite.mean():.3f}  (n={len(gold)})")
print(f"Non-gold mean composite: {non.composite.mean():.3f}  (n={len(non)})")
print(f"Separation:               {gold.composite.mean()-non.composite.mean():+.3f}")

print(f"\nGold-vs-non in composite >= 0.85 (high-confidence kept):")
hi = out_df[out_df.composite >= 0.85]
print(f"  total hi-conf: {len(hi)},  gold: {int(hi.is_gold.sum())},  precision: {hi.is_gold.mean():.1%}")


## Phase 11 — Pass-3 K predictor

In [ ]:
PASS3_PROMPT = """You are a senior Swiss attorney finalizing a legal opinion.

LEGAL QUESTION:
{question}

ISSUE MAP:
{issue_map}

TOP-50 RANKED CANDIDATES (score is your prior 0-1 assessment, 1.0 = must-cite):
{candidates_block}

How many citations should the THOROUGH-but-not-excessive Swiss legal opinion include?
Consider: number of distinct legal issues, question complexity, Swiss legal-writing norms (narrow opinion: 5-12; complex multi-issue: 25-50).

OUTPUT STRICT JSON (no prose):
{{"K": <integer 3-60>, "minimum_per_issue": <integer 1-8>, "reasoning": "<one sentence>"}}
"""

def render_top50(q, top_rows):
    return "\n".join(
        f"  {i:>2}. {r.citation:<28}  score={r.composite:.2f}  issue={r.matched_issue}  role={r.citation_role[:20]}"
        for i, r in enumerate(top_rows.itertuples(), 1)
    )

pass3_prompts, pass3_meta = [], []
for q in QIDS:
    top50 = out_df[out_df.qid == q].sort_values("composite", ascending=False).head(50)
    p = PASS3_PROMPT.format(
        question=qid_to_query[q][:1500],
        issue_map=render_issue_map(qid_to_issues.get(q, [])),
        candidates_block=render_top50(q, top50),
    )
    pass3_prompts.append(p); pass3_meta.append(q)

sp_pass3 = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
                          max_tokens=PASS3_MAX_TOKENS, seed=42)
print(f"Pass-3: {len(pass3_prompts)} prompts ...")
t0 = time.time()
outs3 = llm.chat([[{"role":"user","content":p}] for p in pass3_prompts], sampling_params=sp_pass3)
pass3_raws = [o.outputs[0].text for o in outs3]
print(f"Done in {(time.time()-t0):.1f}s")

del llm; gc.collect()
import torch; torch.cuda.empty_cache()

qid_to_K = {}
qid_to_min_per_issue = {}
for q, raw in zip(pass3_meta, pass3_raws):
    p = parse_json_block(raw) or {}
    qid_to_K[q] = max(3, min(60, int(p.get("K", 15)) if str(p.get("K","")).isdigit() else 15))
    qid_to_min_per_issue[q] = max(1, min(8, int(p.get("minimum_per_issue", 1)) if str(p.get("minimum_per_issue","")).isdigit() else 1))

print(f"\nK predictions:")
for q in QIDS:
    print(f"  {q}: K={qid_to_K[q]}, min_per_issue={qid_to_min_per_issue[q]}  (true gold={gold_totals[q]})")


## Phase 12 — Diversified selection + F1

In [ ]:
def select_picks(q_df, K, min_per_issue, score_floor):
    df = q_df.sort_values("composite", ascending=False).copy()
    df = df[df.composite >= score_floor]
    if len(df) == 0:
        return df.head(0)
    picks = []; picked = set()
    for issue, sub in df.groupby("matched_issue", sort=False):
        if issue in ("none","",""): continue
        for r in sub.head(min_per_issue).itertuples():
            if r.did not in picked:
                picks.append(r); picked.add(r.did)
    remaining = K - len(picks)
    if remaining > 0:
        for r in df.itertuples():
            if r.did in picked: continue
            picks.append(r); picked.add(r.did)
            remaining -= 1
            if remaining <= 0: break
    return pd.DataFrame([{c: getattr(r, c) for c in df.columns} for r in picks[:K]])

def f1(p, r): return 0.0 if (p+r)==0 else 2*p*r/(p+r)

v4 = {"val_001":0.108,"val_002":0.047,"val_003":0.018,"val_004":0.076,"val_005":0.067,
      "val_006":0.109,"val_007":0.066,"val_008":0.013,"val_009":0.000,"val_010":0.065}
v53 = {"val_001":0.167,"val_002":0.132,"val_003":0.064,"val_004":0.100,"val_005":0.091,
       "val_006":0.056,"val_007":0.000,"val_008":0.100,"val_009":0.286,"val_010":0.080}

per_query = {}
all_picks_rows = []
print(f"{'qid':<8} {'gold':>5} {'in_topk':>8} {'K':>4} {'picks':>6} {'correct':>8} "
      f"{'P':>5} {'R':>5} {'F1':>5}  {'v4':>5}  {'v5.3':>5}  dv4    dv5.3")
for q in QIDS:
    q_df = out_df[out_df.qid == q]
    K = qid_to_K[q]; mp = qid_to_min_per_issue[q]
    picks_df = select_picks(q_df, K, mp, SCORE_FLOOR)
    pick_dids = set(picks_df.did) if "did" in picks_df.columns else set()
    correct = pick_dids & qid_to_gold[q]
    p = len(correct)/max(1, len(pick_dids))
    r = len(correct)/max(1, gold_totals[q])
    f1_v = f1(p, r)
    per_query[q] = {"gold": gold_totals[q], "gold_in_topk": int(sb_per_query[q].is_gold.sum()),
                    "K_predicted": K, "picks": len(pick_dids), "correct": len(correct),
                    "P": p, "R": r, "F1": f1_v}
    d4, d53 = f1_v - v4[q], f1_v - v53[q]
    s4 = "+" if d4 > 0 else ""; s53 = "+" if d53 > 0 else ""
    print(f"  {q:<6} {gold_totals[q]:>5} {int(sb_per_query[q].is_gold.sum()):>8} "
          f"{K:>4} {len(pick_dids):>6} {len(correct):>8} "
          f"{p:>.3f} {r:>.3f} {f1_v:>.3f}  {v4[q]:>.3f}  {v53[q]:>.3f}  "
          f"{s4}{d4:+.3f}  {s53}{d53:+.3f}")
    for _, row in picks_df.iterrows():
        rd = row.to_dict()
        rd["correct"] = rd.get("did","") in qid_to_gold[q]
        all_picks_rows.append(rd)

macro_P = sum(per_query[q]["P"] for q in QIDS)/10
macro_R = sum(per_query[q]["R"] for q in QIDS)/10
macro_F = sum(per_query[q]["F1"] for q in QIDS)/10
print(f"\nMACRO  P={macro_P:.3f}  R={macro_R:.3f}  F1={macro_F:.3f}")
print(f"\nBaselines:  v4 macro F1=0.057  |  v5.3 macro K-cap F1=0.107  |  v6 projected ~0.45")
print(f"v7 macro F1 delta vs v4:    {macro_F-0.057:+.3f}")
print(f"v7 macro F1 delta vs v5.3:  {macro_F-0.107:+.3f}")


## Phase 13 — Save outputs

In [ ]:
out_df.to_parquet(OUT_DIR / "v7_scored.parquet", index=False)
if all_picks_rows:
    pd.DataFrame(all_picks_rows).to_parquet(OUT_DIR / "v7_picks.parquet", index=False)
with open(OUT_DIR / "v7_issue_maps.json", "w", encoding="utf-8") as f:
    json.dump(qid_to_issues, f, indent=2, ensure_ascii=False)
with open(OUT_DIR / "v7_expanded_landscapes.json", "w", encoding="utf-8") as f:
    json.dump({q: sorted(qid_to_expanded[q]) for q in QIDS}, f, indent=2, ensure_ascii=False)
with open(OUT_DIR / "v7_metrics.json", "w") as f:
    json.dump({
        "config": {"TOP_K": TOP_K, "PASS1_SEEDS": PASS1_SEEDS,
                   "SCORE_FLOOR": SCORE_FLOOR,
                   "TOP_PRECEDENTS_PER_STATUTE": TOP_PRECEDENTS_PER_STATUTE,
                   "BIB_NEIGHBORS_PER_STATUTE": BIB_NEIGHBORS_PER_STATUTE,
                   "EXPANDED_LANDSCAPE_CAP": EXPANDED_LANDSCAPE_CAP},
        "macro_P": macro_P, "macro_R": macro_R, "macro_F1": macro_F,
        "per_query": per_query,
        "qid_to_K": qid_to_K,
        "qid_to_min_per_issue": qid_to_min_per_issue,
        "v4_macro_F1": 0.057, "v53_macro_K_cap_F1": 0.107,
    }, f, indent=2)
print(f"Saved 5 files to {OUT_DIR}")
print(f"\nv7 macro F1: {macro_F:.3f}  (v4=0.057, v5.3=0.107)")
